In [3]:
from preprocessing import Preprocessor
from analysis import Analysis, EIS_Analysis
import pandas as pd


In [4]:
filepath = '/Users/maxtenenbaum/Desktop/20240717_IF0003_IIA31_B_PEDOT_E01_EIS.DTA'

In [5]:
# Loads filepath into preprocessor
preprocessing = Preprocessor(filepath)

# Loading EIS analysis class
eis_analysis = EIS_Analysis()


In [6]:
# Checks if EIS
preprocessing.pull_eis_metadata()

In [7]:
def new_extract_eis():
        try:
            with open(filepath, 'r', encoding='utf-8') as file:
                lines = file.readlines()
        except UnicodeDecodeError:
            with open(filepath, 'r', encoding='ISO-8859-1') as file:
                lines = file.readlines()

        eis_data = []
        headers = []
        data_section = False

        for i, line in enumerate(lines):
            if 'ZCURVE' in line:
                data_section = True
                continue
            if data_section and line.startswith('\tPt'):
                headers = line.strip().split('\t')[1:]  
                data_section = False  
            elif line.startswith('\t'):
                values = line.strip().split('\t')[1:]
                if len(values) == len(headers):
                    eis_data.append(values)

        eis_df = pd.DataFrame(eis_data, columns=headers)
        eis_df = eis_df.apply(pd.to_numeric, errors='ignore')
        #eis_dataframe = eis_df[['Freq', 'Zreal', 'Zmod', 'Zphz']]

        return eis_df

In [8]:
dataframe = new_extract_eis()

In [9]:
print(dataframe)

    Time      Freq     Zreal      Zimag  Zsig      Zmod       Zphz  \
0   None      None      None       None  None      None       None   
1      s        Hz       ohm        ohm     V       ohm          °   
2      3  100019.5  5024.838  -255.4786     1  5031.328  -2.910592   
3      4  72011.72  5046.278  -257.4408     1   5052.84  -2.920468   
4      6  51855.47  5074.711  -276.0874     1  5082.216  -3.114081   
5      7  37324.22  5108.416  -312.8321     1  5117.985  -3.504335   
6      9  26894.53  5181.104  -267.9436     1  5188.028  -2.960445   
7     11  19394.53  5218.175  -355.7101     1  5230.285  -3.899679   
8     12  13831.02  5272.947  -462.7339     1  5293.212  -5.015213   
9     14  10019.53   5339.57  -585.3713     1  5371.562   -6.25629   
10    15  7207.031  5426.975  -734.3221     1   5476.43  -7.705872   
11    17  5150.463  5537.825  -920.6447     1   5613.83  -9.438905   
12    18  3715.701  5667.281  -1143.429     1  5781.479  -11.40684   
13    20  2686.738  

In [10]:
frequencies = [1.0, 1000.0, 100000.0, 1000000.0]
closest_frequencies = eis_analysis.get_closest_freq(dataframe, frequencies)
filtered_df = dataframe[dataframe["Freq"].isin(closest_frequencies)]

In [11]:
print(filtered_df)

   Time           Freq     Zreal      Zimag Zsig      Zmod       Zphz  \
2     3  100019.500000  5024.838  -255.4786    1  5031.328  -2.910592   
16   25    1000.702000   6551.32  -2748.913    1  7104.668  -22.76277   
37   68       0.997765  177734.8  -323842.6    1  369409.9  -61.24066   

               Idc        Vdc IERange           Imod       Vmod      Temp  
2   -4.892301E-008  0.2033607       8  1.943454E-006  0.0098018  1399.337  
16  -4.723562E-009   0.203403       7  1.409053E-006  0.0100121  1399.162  
37  -8.882289E-011  0.2033564       5  2.714566E-008   0.010027  1399.337  


In [19]:
new_df = pd.DataFrame(' ', index=dataframe.index, columns=dataframe.columns)
new_df.iloc[:len(filtered_df)] = filtered_df.values

In [21]:
print(new_df)

   Time      Freq     Zreal      Zimag Zsig      Zmod       Zphz  \
0     3  100019.5  5024.838  -255.4786    1  5031.328  -2.910592   
1    25  1000.702   6551.32  -2748.913    1  7104.668  -22.76277   
2    68  0.997765  177734.8  -323842.6    1  369409.9  -61.24066   
3                                                                  
4                                                                  
5                                                                  
6                                                                  
7                                                                  
8                                                                  
9                                                                  
10                                                                 
11                                                                 
12                                                                 
13                                              

In [14]:

new_df['Zmod_Full'] = dataframe['Zmod'].iloc[2:].reset_index(drop=True)
new_df['Zphz_Full'] = dataframe['Zphz'].iloc[2:].reset_index(drop=True)


In [20]:
print(new_df.shape)

(38, 13)
